In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.cluster import DBSCAN, HDBSCAN
from sklearn.preprocessing import StandardScaler

from src.analysis.dim_reducer import reduce_dimensionality
from src.util.datasets import load_mnist_dataset

DATASET_PATH = Path("datasets/wine_quality/wine+quality/winequality-red.csv")
df = pd.read_csv(DATASET_PATH, sep=";")
df = df.reset_index(drop=True)
df["row_id"] = df.index

In [ ]:
X = df.drop(columns=["quality", "row_id"]).values
scaler = StandardScaler()
X_scaled: pd.DataFrame = scaler.fit_transform(X)

In [ ]:
mnist_data = load_mnist_dataset()
# mnist_x_combined = np.concatenate([mnist_data[0], mnist_data[2]], axis=0)
mnist_x_combined = mnist_data[0]  # only use training data for now
mnist_df = pd.DataFrame(mnist_x_combined, columns=[f"pixel_{i}" for i in range(784)])
# mnist_df["label"] = mnist_data[1] + mnist_data[3]
mnist_df["label"] = mnist_data[1]
mnist_df["row_id"] = mnist_df.index
mnist_x_scaled = scaler.fit_transform(mnist_x_combined)
mnist_X = mnist_df.drop(columns=["label", "row_id"]).values


In [ ]:
import umap
reducer = umap.UMAP(n_components=10, n_neighbors=30, min_dist=0.0)
X_umap = reducer.fit_transform(X_scaled)
model = HDBSCAN(min_cluster_size=15, min_samples=5)
labels = model.fit_predict(X_umap)
df["cluster"] = labels

In [ ]:
import umap
from sklearn.manifold import MDS

reducer = umap.UMAP(n_components=10, n_neighbors=30, min_dist=0.0)
X_umap = reducer.fit_transform(X_scaled)
model = HDBSCAN(min_cluster_size=15, min_samples=5)
labels = model.fit_predict(X_umap)
df["cluster"] = labels

# exclude HDBSCAN noise points
mask = df["cluster"] != -1
feature_cols = df.columns.drop(["quality", "row_id", "cluster"])

X_scaled_df = pd.DataFrame(X_scaled, columns=feature_cols, index=df.index)
centroids = X_scaled_df[mask].groupby(df.loc[mask, "cluster"]).mean()

# 2D layout that preserves pairwise centroid distances
mds = MDS(n_components=2, dissimilarity="euclidean",
          random_state=42, n_init=8, normalized_stress="auto")
centroids_2d = mds.fit_transform(centroids.values)

sizes = df.loc[mask, "cluster"].value_counts().sort_index()

layout_df = pd.DataFrame({
    "x": centroids_2d[:, 0],
    "y": centroids_2d[:, 1],
    "cluster": centroids.index,
    "size": sizes.values,
})


fig = px.scatter(layout_df, x="x", y="y", size="size", color="size", hover_data=["cluster"],
                 title="HDBSCAN Clusters of Wine Quality Dataset (MDS Layout)")

fig.update_yaxes(scaleanchor="x", scaleratio=1)
fig.update_layout(legend_title_text="Cluster")
fig.show()

In [ ]:
from sklearn.decomposition import PCA
from scipy.stats import gaussian_kde

fig = go.Figure()

for i, c in enumerate(centroids.index):
    pts = X_scaled_df.loc[df["cluster"] == c].values
    if len(pts) < 5:
        continue

    # local 2D embedding of just this cluster
    pca = PCA(n_components=2)
    pts_2d = pca.fit_transform(pts)

    # KDE on the local 2D points
    kde = gaussian_kde(pts_2d.T, bw_method="scott")
    pad = 0.5 * pts_2d.std(axis=0).max()
    lx_min, lx_max = pts_2d[:, 0].min() - pad, pts_2d[:, 0].max() + pad
    ly_min, ly_max = pts_2d[:, 1].min() - pad, pts_2d[:, 1].max() + pad
    lx = np.linspace(lx_min, lx_max, 60)
    ly = np.linspace(ly_min, ly_max, 60)
    LX, LY = np.meshgrid(lx, ly)
    Z = kde(np.vstack([LX.ravel(), LY.ravel()])).reshape(LX.shape)

    # scale the local footprint and place at the cluster's MDS position
    cx, cy = centroids_2d[i]
    local_extent = max(lx_max - lx_min, ly_max - ly_min)
    target_size = 0.8 * np.sqrt(sizes.loc[c] / sizes.max()) + 0.3  # tunable
    scale = target_size / local_extent

    plot_x = (lx - (lx_min + lx_max) / 2) * scale + cx
    plot_y = (ly - (ly_min + ly_max) / 2) * scale + cy

    fig.add_trace(go.Contour(
        x=plot_x, y=plot_y, z=Z,
        colorscale="Viridis", showscale=False,
        contours=dict(coloring="heatmap", showlines=True,
                      start=Z.max() * 0.05, size=Z.max() * 0.15),
        line_smoothing=0.85, hoverinfo="skip",
    ))

fig.add_trace(go.Scatter(
    x=centroids_2d[:, 0], y=centroids_2d[:, 1],
    mode="text",
    text=[f"C{c}" for c in centroids.index],
    textposition="top center",
    showlegend=False,
))

fig.update_yaxes(scaleanchor="x", scaleratio=1)
fig.update_layout(
    title="Cluster topography — global MDS layout, local KDE per cluster",
    plot_bgcolor="white",
)
fig.show()

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text
from plotly.subplots import make_subplots
import plotly.graph_objects as go

feature_cols = df.columns.drop(["quality", "row_id", "cluster"])

def cluster_characteristics(cluster_id, df, X_scaled_df, feature_cols,
                            top_n=8, tree_depth=3):
    in_cluster = df["cluster"] == cluster_id
    pts = X_scaled_df.loc[in_cluster, feature_cols]

    # in scaled space, global mean=0 and global std=1, so:
    z_mean = pts.mean()   # signed z-score of cluster mean per dim
    z_std  = pts.std()    # within-cluster std, in units of global std
    order = z_mean.abs().sort_values(ascending=False).index.tolist()
    z_mean, z_std = z_mean[order], z_std[order]

    R = max(3.0, z_mean.abs().max() + 1)   # radial extent in z-units

    fig = make_subplots(rows=1, cols=2,
        specs=[[{"type": "polar"}, {"type": "xy"}]],
        column_widths=[0.55, 0.45],
        subplot_titles=(f"Cluster {cluster_id} profile",
                        "Top distinguishing dimensions"))

    theta = order + [order[0]]

    # reference ring at global mean (z=0 → r=R)
    fig.add_trace(go.Scatterpolar(
        r=[R]*len(theta), theta=theta, mode="lines",
        line=dict(color="gray", dash="dot"),
        name="global mean", hoverinfo="skip"), row=1, col=1)

    # within-cluster ±1σ band (the "uncertainty" of the cluster on each dim)
    band_hi = (z_mean + z_std).tolist() + [(z_mean + z_std).iloc[0]]
    band_lo = (z_mean - z_std).tolist() + [(z_mean - z_std).iloc[0]]
    fig.add_trace(go.Scatterpolar(r=[v+R for v in band_hi], theta=theta,
        mode="lines", line=dict(width=0), showlegend=False,
        hoverinfo="skip"), row=1, col=1)
    fig.add_trace(go.Scatterpolar(r=[v+R for v in band_lo], theta=theta,
        fill="tonext", fillcolor="rgba(70,130,200,0.25)",
        mode="lines", line=dict(width=0),
        name="±1σ within-cluster"), row=1, col=1)

    # cluster mean polygon — outside ring = above global, inside = below
    means = z_mean.tolist() + [z_mean.iloc[0]]
    fig.add_trace(go.Scatterpolar(r=[v+R for v in means], theta=theta,
        mode="lines+markers",
        line=dict(color="rgb(50,90,180)", width=2),
        name="cluster mean"), row=1, col=1)

    fig.update_polars(radialaxis=dict(range=[0, 2*R], showticklabels=False))

    # signed-z bar chart for the top-N most distinguishing dims
    top = z_mean.head(top_n)
    fig.add_trace(go.Bar(
        x=top.values, y=top.index, orientation="h",
        marker_color=["crimson" if v < 0 else "steelblue" for v in top.values],
        text=[f"within σ={z_std[d]:.2f}" for d in top.index],
        textposition="outside", showlegend=False), row=1, col=2)
    fig.update_xaxes(title="z-score of cluster mean (vs. global)", row=1, col=2)
    fig.update_yaxes(autorange="reversed", row=1, col=2)
    fig.update_layout(
        title=f"Cluster {cluster_id}  •  n={in_cluster.sum()}", height=520)

    # predicate rules — train on ORIGINAL units so thresholds are interpretable
    tree = DecisionTreeClassifier(max_depth=tree_depth,
                                  class_weight="balanced", random_state=0)
    tree.fit(df[feature_cols].values, in_cluster.astype(int).values)
    rules = export_text(tree, feature_names=list(feature_cols))

    return fig, rules

In [ ]:
for c in sorted(df["cluster"].unique()):
    if c == -1:
        continue  # skip outliers
    fig, rules = cluster_characteristics(c, df, X_scaled_df, feature_cols)
    fig.show()
    print(f"\n--- Cluster {c} predicates ---\n{rules}")